# 11 工程化：配置、流水线、实验记录与交易信号

## 11.1 本章目标

前面章节已经分别跑通数据、因子、组合、回测和绩效评估。本章把这些步骤收拢成一个可重复运行的策略流水线。

学完本章后，你应该能：

- 用配置记录策略参数。
- 为每次实验生成稳定的输出目录。
- 调用 `lib.pipeline.run_etf_strategy_pipeline` 跑完整 ETF 策略。
- 用 `latest_target_weights` 和 `target_weights_to_orders` 生成目标权重和订单建议。
- 区分“订单建议/交易信号”和真实券商下单。

## 11.2 前置条件

`04-10` 章已经讲过流水线中的每个组件。本章不重新解释每个公式，而是关注如何把它们组织成可复现的工程流程。

## 11.3 学习路线

1. 先手写一个最小实验配置。
2. 读取项目配置文件。
3. 演示目标权重如何转成订单建议。
4. 调用 `lib.pipeline.run_etf_strategy_pipeline` 跑完整流程。
5. 检查并保存 `outputs/results` 下的实验记录。


In [1]:
from pathlib import Path
import sys
import json
from copy import deepcopy


def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    candidates += [candidate / "pyquant-roadmap" for candidate in candidates]
    for candidate in candidates:
        if (candidate / "lib").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise RuntimeError("Cannot find pyquant-roadmap project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import yaml

from lib.paths import CONFIG_DIR, RESULTS_DIR
from lib.pipeline import run_etf_strategy_pipeline
from lib.trading import latest_target_weights, target_weights_to_orders

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pd.Series({"project_root": ".", "results_dir": RESULTS_DIR.relative_to(PROJECT_ROOT).as_posix()}, name="value")


project_root                  .
results_dir     outputs/results
Name: value, dtype: object

## 11.4 手写最小实验记录

一次实验至少要记录三类信息：

- **参数**：数据区间、调仓频率、TopN、成本、资金规模。
- **输入**：使用的数据源和缓存路径。
- **输出**：结果目录、报告文件、信号文件。

这些记录不复杂，但能避免“跑完一次之后不知道自己做过什么”。


In [2]:
manual_config = {
    "start_date": "2021-01-01",
    "end_date": "2023-12-31",
    "asset_type": "ETF",
    "rebalance_freq": "M",
    "top_n": 3,
    "cost_bps": 8,
    "capital": 1_000_000,
    "lot_size": 100,
    "factor_weights": {
        "momentum_60": 0.45,
        "low_vol_20": 0.35,
        "ma_gap_20_60": 0.20,
    },
}

manual_experiment_id = "manual_etf_top3_monthly_20210101_20231231"
manual_output_dir = RESULTS_DIR / "chapter11_engineering_pipeline" / manual_experiment_id
manual_record = {
    "experiment_id": manual_experiment_id,
    "config": manual_config,
    "input_data": "data/sample/*.parquet",
    "output_dir": str(manual_output_dir.relative_to(PROJECT_ROOT)).replace("\\", "/"),
    "status": "planned",
}

parameter_preview = pd.DataFrame(
    [{"name": key, "value": value} for key, value in manual_config.items() if key != "factor_weights"]
)
display(parameter_preview)
display(pd.Series(manual_record, name="manual_experiment_record"))


,name,value
0,start_date,2021-01-01
1,end_date,2023-12-31
2,asset_type,ETF
3,rebalance_freq,M
4,top_n,3
5,cost_bps,8
6,capital,1000000
7,lot_size,100


experiment_id            manual_etf_top3_monthly_20210101_20231231
config           {'start_date': '2021-01-01', 'end_date': '2023...
input_data                                   data/sample/*.parquet
output_dir       outputs/results/chapter11_engineering_pipeline...
status                                                     planned
Name: manual_experiment_record, dtype: object

## 11.5 导入依赖

工程化不是把所有代码塞进 `lib/`。notebook 仍然负责展示流程和解释结果；`pandas` 用来整理表格，`PyYAML` 读取配置，`json` 和 `pathlib` 管理实验记录与路径。

本章的策略内核来自 `lib.factors` 中的 `momentum_60`、`low_vol_20`、`ma_gap_20_60`，这些因子都采用“先滞后一日再参与调仓”的约定，避免同日收盘信息泄漏。


In [3]:
parameter_notes = pd.DataFrame(
    [
        ("start_date / end_date", "样本开始和结束日期"),
        ("asset_type", "从资产元数据中筛选哪类资产，这里是 ETF"),
        ("rebalance_freq", "调仓频率，M 表示月度，Q 表示季度"),
        ("top_n", "每次调仓选择分数最高的 ETF 数量"),
        ("factor_weights", "多因子合成分数时的权重"),
        ("cost_bps", "单边成本假设，8 bps 表示 0.08%"),
        ("capital", "用于订单估算的资金规模"),
        ("lot_size", "最小交易单位，A 股 ETF 通常按 100 份取整"),
        ("current_positions", "当前真实持仓，决定订单建议的买卖数量"),
        ("output_dir", "本次实验的独立输出目录"),
    ],
    columns=["parameter", "meaning"],
)
parameter_notes


,parameter,meaning
0,start_date / end_date,样本开始和结束日期
1,asset_type,从资产元数据中筛选哪类资产，这里是 ETF
2,rebalance_freq,调仓频率，M 表示月度，Q 表示季度
3,top_n,每次调仓选择分数最高的 ETF 数量
4,factor_weights,多因子合成分数时的权重
5,cost_bps,单边成本假设，8 bps 表示 0.08%
6,capital,用于订单估算的资金规模
7,lot_size,最小交易单位，A 股 ETF 通常按 100 份取整
8,current_positions,当前真实持仓，决定订单建议的买卖数量
9,output_dir,本次实验的独立输出目录


## 11.6 读取项目配置并生成实验目录

配置文件 `configs/strategy_demo.yml` 保存主线策略参数。实验 ID 用关键参数拼出来，保证不同参数组合会写入不同目录，便于复盘和比较。


In [4]:
cfg_path = CONFIG_DIR / "strategy_demo.yml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))


def compact_date(value: str) -> str:
    return str(value).replace("-", "")


def build_experiment_id(config: dict) -> str:
    freq = str(config["rebalance_freq"]).lower()
    start = compact_date(config["start_date"])
    end = compact_date(config["end_date"])
    cost = str(config["cost_bps"]).replace(".", "p")
    return f"etf_top{config['top_n']}_{freq}_{start}_{end}_cost{cost}bps"


EXPERIMENT_ID = build_experiment_id(cfg)
EXPERIMENT_DIR = RESULTS_DIR / "chapter11_engineering_pipeline" / EXPERIMENT_ID
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

io_map = pd.DataFrame(
    [
        {"kind": "config", "path": str(cfg_path.relative_to(PROJECT_ROOT)).replace("\\", "/")},
        {"kind": "output_dir", "path": str(EXPERIMENT_DIR.relative_to(PROJECT_ROOT)).replace("\\", "/")},
        {"kind": "record", "path": str((EXPERIMENT_DIR / "experiment_record.json").relative_to(PROJECT_ROOT)).replace("\\", "/")},
    ]
)

display(pd.Series(cfg, name="value").to_frame())
io_map


,value
start_date,2021-01-01
end_date,2023-12-31
top_n,3
rebalance_freq,M
cost_bps,8
capital,1000000
lot_size,100
asset_type,ETF
factor_weights,"{'momentum_60': 0.45, 'low_vol_20': 0.35, 'ma_..."


,kind,path
0,config,configs/strategy_demo.yml
1,output_dir,outputs/results/chapter11_engineering_pipeline...
2,record,outputs/results/chapter11_engineering_pipeline...


## 11.7 手写目标权重到订单建议

`latest_target_weights(weight_matrix)` 读取最后一个交易日的目标权重。

`target_weights_to_orders(weight_matrix, price_matrix, capital, current_positions, lot_size)` 会把目标权重、最新价格和当前持仓转换成订单建议，并按 `lot_size` 做手数取整。

注意：这里输出的是学习用订单建议，不是可以直接发送给券商的真实订单。真实交易还要检查账户、流动性、权限、风控和合规要求。


In [5]:
toy_dates = pd.to_datetime(["2024-01-31", "2024-02-29"])
toy_target_weights = pd.DataFrame(
    {
        "510300": [0.40, 0.50],
        "510500": [0.40, 0.00],
        "159915": [0.20, 0.50],
    },
    index=toy_dates,
)
toy_prices = pd.DataFrame(
    {
        "510300": [3.90, 4.00],
        "510500": [5.80, 6.00],
        "159915": [1.95, 2.00],
    },
    index=toy_dates,
)
toy_current_positions = {"510300": 2_000, "510500": 1_000, "159915": 0}

toy_latest = latest_target_weights(toy_target_weights)
toy_orders = target_weights_to_orders(
    toy_target_weights,
    toy_prices,
    capital=100_000,
    current_positions=toy_current_positions,
    lot_size=100,
)

display(toy_latest)
display(toy_orders)


,date,code,target_weight
0,2024-02-29,159915,0.5
1,2024-02-29,510300,0.5


,signal_date,code,side,price,target_weight,current_weight,delta_weight,target_shares,current_shares,order_shares,order_value
0,2024-02-29,159915,BUY,2.0,0.5,0.00,0.50,25000,0,25000,50000.0
1,2024-02-29,510300,BUY,4.0,0.5,0.08,0.42,12500,2000,10500,42000.0
2,2024-02-29,510500,SELL,6.0,0.0,0.06,-0.06,0,1000,-1000,-6000.0


## 11.8 运行完整 ETF 流水线

现在把前面章节串起来，调用 `run_etf_strategy_pipeline`：

- 从 `data/sample` 读取缓存 ETF 数据。
- 用 `lib.factors` 构建并合成多因子分数。
- 按 `rebalance_freq` 生成调仓日。
- 选择 `top_n` 只 ETF 并生成权重。
- 回测、评估、保存报告，并写入 `EXPERIMENT_DIR`。

本章默认 `CURRENT_POSITIONS = {}`，代表从空仓开始生成订单建议。真实使用时必须替换成账户实际持仓。


In [6]:
CURRENT_POSITIONS = {}

result = run_etf_strategy_pipeline(
    top_n=cfg["top_n"],
    cost_bps=cfg["cost_bps"],
    capital=cfg["capital"],
    lot_size=cfg["lot_size"],
    factor_weights=cfg["factor_weights"],
    asset_type=cfg["asset_type"],
    output_dir=EXPERIMENT_DIR,
    start_date=cfg["start_date"],
    end_date=cfg["end_date"],
    rebalance_freq=cfg["rebalance_freq"],
    current_positions=CURRENT_POSITIONS,
)

list(result.keys())


['prices',
 'factors',
 'sparse_rebalance_weights',
 'target_weights',
 'latest_target_weights',
 'orders',
 'report',
 'bt_result',
 'assets']

## 11.9 查看最新目标权重和订单建议

`latest_target_weights` 告诉你策略最新希望持有什么；`orders` 告诉你从当前持仓走向目标持仓需要买卖什么。

这两张表是“策略输出”，不是“投资建议”。


In [7]:
latest = result["latest_target_weights"].copy()
orders = result["orders"].copy()

latest_display = latest.copy()
latest_display["date"] = pd.to_datetime(latest_display["date"]).dt.date
orders_display = orders.copy()
if not orders_display.empty:
    orders_display["signal_date"] = pd.to_datetime(orders_display["signal_date"]).dt.date

latest_display = latest_display.sort_values("target_weight", ascending=False)

display(latest_display.round({"target_weight": 4, "score": 4}))
display(orders_display.round({"price": 4, "target_weight": 4, "current_weight": 4, "delta_weight": 4, "order_value": 2}))


,date,code,target_weight,score
0,2023-12-29,510300,0.3333,-0.7652
1,2023-12-29,510500,0.3333,0.6667
2,2023-12-29,512100,0.3333,0.8990


,signal_date,code,side,price,target_weight,current_weight,delta_weight,target_shares,current_shares,order_shares,order_value
0,2023-12-29,510300,BUY,3.219,0.3333,0.0,0.3333,103500,0,103500,333166.5
1,2023-12-29,510500,BUY,5.279,0.3333,0.0,0.3333,63100,0,63100,333104.9
2,2023-12-29,512100,BUY,2.296,0.3333,0.0,0.3333,145100,0,145100,333149.6


## 11.10 读懂订单建议字段

常用字段含义如下：

- `target_weight`：策略目标权重。
- `side`：买入、卖出或保持不动。
- `order_shares`：按手数取整后的交易数量。
- `order_value`：按最新价格估算的交易金额。

如果真实执行，还需要检查交易时间、报价、涨跌停、资金可用性和券商接口限制。


In [8]:
order_column_notes = pd.DataFrame(
    [
        ("signal_date", "信号对应的最新交易日期"),
        ("code", "ETF 代码"),
        ("side", "BUY / SELL，表示买入或卖出方向"),
        ("price", "用于估算订单价值的最新价格"),
        ("target_weight", "策略目标权重"),
        ("current_weight", "按当前持仓估算的现有权重"),
        ("delta_weight", "目标权重和当前权重的差"),
        ("target_shares", "按 lot_size 取整后的目标份额"),
        ("current_shares", "当前持有份额"),
        ("order_shares", "建议交易份额，正数买入，负数卖出"),
        ("order_value", "按最新价格估算的交易金额"),
    ],
    columns=["column", "meaning"],
)
order_column_notes


,column,meaning
0,signal_date,信号对应的最新交易日期
1,code,ETF 代码
2,side,BUY / SELL，表示买入或卖出方向
3,price,用于估算订单价值的最新价格
4,target_weight,策略目标权重
5,current_weight,按当前持仓估算的现有权重
6,delta_weight,目标权重和当前权重的差
7,target_shares,按 lot_size 取整后的目标份额
8,current_shares,当前持有份额
9,order_shares,建议交易份额，正数买入，负数卖出


## 11.11 保存实验记录

除了 CSV、图表和 HTML 报告，本章还会保存 `experiment_record.json`。它把配置、输出路径、样本行数和最新信号日期记录下来，方便以后复盘。


In [9]:
def rel_path(path: Path) -> str:
    return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve())).replace("\\", "/")

asset_paths = {name: rel_path(path) for name, path in result["assets"].items()}
record_path = EXPERIMENT_DIR / "experiment_record.json"

latest_signal_date = None if latest.empty else str(pd.to_datetime(latest["date"].max()).date())
experiment_record = {
    "experiment_id": EXPERIMENT_ID,
    "config_path": rel_path(cfg_path),
    "config": cfg,
    "current_positions": CURRENT_POSITIONS,
    "outputs": {**asset_paths, "experiment_record": rel_path(record_path)},
    "row_counts": {
        "prices": int(len(result["prices"])),
        "factors": int(len(result["factors"])),
        "target_weight_dates": int(len(result["target_weights"])),
        "orders": int(len(orders)),
    },
    "latest_signal_date": latest_signal_date,
    "latest_target_weight_sum": float(latest["target_weight"].sum()) if not latest.empty else 0.0,
    "estimated_order_value_abs": float(orders["order_value"].abs().sum()) if not orders.empty else 0.0,
}

record_path.write_text(json.dumps(experiment_record, ensure_ascii=False, indent=2), encoding="utf-8")

display(pd.DataFrame([{"artifact": name, "path": path} for name, path in experiment_record["outputs"].items()]))
experiment_record["row_counts"]


,artifact,path
0,nav_curve,outputs/results/chapter11_engineering_pipeline...
1,drawdown_curve,outputs/results/chapter11_engineering_pipeline...
2,metrics_csv,outputs/results/chapter11_engineering_pipeline...
3,metrics_md,outputs/results/chapter11_engineering_pipeline...
4,target_weights,outputs/results/chapter11_engineering_pipeline...
5,trade_orders,outputs/results/chapter11_engineering_pipeline...
6,benchmark_comparison,outputs/results/chapter11_engineering_pipeline...
7,quantstats_report,outputs/results/chapter11_engineering_pipeline...
8,factor_scores,outputs/results/chapter11_engineering_pipeline...
9,strategy_returns,outputs/results/chapter11_engineering_pipeline...


{'prices': 2900, 'factors': 2656, 'target_weight_dates': 725, 'orders': 3}

## 11.12 质量检查

流水线运行后至少检查三件事：

- 目标权重总和不超过 100%。
- 订单股数符合最小交易单位。
- 关键输出文件都已经写入实验目录。

这些检查不能保证策略好，但能减少低级工程错误。


In [10]:
target_weight_sum_max = float(result["target_weights"].sum(axis=1).max())
latest_weight_sum = float(latest["target_weight"].sum()) if not latest.empty else 0.0
lot_aligned = orders.empty or bool((orders["order_shares"].abs() % int(cfg["lot_size"])).eq(0).all())
all_assets_exist = all(Path(path).exists() for path in result["assets"].values()) and record_path.exists()
outputs_under_experiment_dir = all(
    str(Path(path).resolve()).startswith(str(EXPERIMENT_DIR.resolve()))
    for path in list(result["assets"].values()) + [record_path]
)

checks = pd.Series(
    {
        "target_weight_sum_max": target_weight_sum_max,
        "latest_weight_sum": latest_weight_sum,
        "lot_aligned": lot_aligned,
        "all_assets_exist": all_assets_exist,
        "outputs_under_experiment_dir": outputs_under_experiment_dir,
        "factor_dates_min": str(pd.to_datetime(result["factors"]["date"].min()).date()),
        "factor_dates_max": str(pd.to_datetime(result["factors"]["date"].max()).date()),
    },
    name="value",
)
display(checks.to_frame())

assert target_weight_sum_max <= 1.0 + 1e-9
assert latest_weight_sum <= 1.0 + 1e-9
assert lot_aligned
assert all_assets_exist
assert outputs_under_experiment_dir


,value
target_weight_sum_max,1.0
latest_weight_sum,1.0
lot_aligned,True
all_assets_exist,True
outputs_under_experiment_dir,True
factor_dates_min,2021-04-08
factor_dates_max,2023-12-29


## 11.13 小练习：改参数再跑一次

尝试把 `top_n` 从 3 改为 2，或者把调仓频率从 `M` 改为 `Q`，观察实验 ID、输出目录、最新目标权重和订单建议如何变化。

不要覆盖原实验目录。


In [11]:
exercise_answer = pd.DataFrame(
    [
        ("cfg['top_n']", "从 3 改成 2 后，每次最多持有 2 只 ETF"),
        ("cfg['rebalance_freq']", "从 M 改成 Q 后，调仓频率从月度变成季度"),
        ("EXPERIMENT_ID / EXPERIMENT_DIR", "实验 ID 和输出目录会随参数变化，避免覆盖旧结果"),
        ("latest_target_weights", "最新目标持仓中的 ETF 数量和权重可能变化"),
        ("orders", "订单建议会根据新目标权重重新计算"),
        ("experiment_record.json", "会记录本次实验的参数、输出路径和行数"),
    ],
    columns=["object", "expected_change"],
)
exercise_answer


,object,expected_change
0,cfg['top_n'],从 3 改成 2 后，每次最多持有 2 只 ETF
1,cfg['rebalance_freq'],从 M 改成 Q 后，调仓频率从月度变成季度
2,EXPERIMENT_ID / EXPERIMENT_DIR,实验 ID 和输出目录会随参数变化，避免覆盖旧结果
3,latest_target_weights,最新目标持仓中的 ETF 数量和权重可能变化
4,orders,订单建议会根据新目标权重重新计算
5,experiment_record.json,会记录本次实验的参数、输出路径和行数


## 11.14 本章小结

本章把研究流程变成了可复现流水线：配置决定参数，流水线串联数据、因子、组合、回测、报告和信号，实验记录保存关键上下文。

第 12 章会暂时跳出主线代码，从策略分类角度理解：你刚刚完成的 ETF 多因子 TopN 策略，和常见趋势、均值回归、横截面动量策略分别属于什么类型。
